# SimCLR Baseline — CIFAR-10 Screening
**Stage I:** 200 epochs, ResNet-18, batch 256, seed 42

Expected runtime: ~2–4h on T4. Fits comfortably in one 12h session.

In [ ]:
import torch
print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

In [ ]:
import os, subprocess, glob, shutil

REPO_URL = "https://github.com/SAlaMusa/IDL.git"
REPO_DIR = "/kaggle/working/IDL"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

In [ ]:
# Verify all imports before training
import sys
sys.path.insert(0, REPO_DIR)
from models.resnet_simclr import ResNetSimCLR
from optimizers.lars import LARS
print("Imports OK")

## Smoke Test (5 epochs)

In [ ]:
result = subprocess.run([
    "python", "run.py",
    "--config", "configs/baseline_cifar10.yaml",
    "--epochs", "5", "--batch-size", "128", "-j", "2", "--seed", "42",
], capture_output=False)

if result.returncode != 0:
    raise RuntimeError("Smoke test failed — fix error above before continuing.")
print("Smoke test PASSED.")

## CIFAR-10 Pretraining — 200 epochs

In [ ]:
result = subprocess.run([
    "python", "run.py",
    "--config", "configs/baseline_cifar10.yaml",
    "--epochs", "200", "--batch-size", "256", "-j", "2", "--seed", "42",
], capture_output=False)

if result.returncode != 0:
    raise RuntimeError("Pretraining failed.")

# Find and persist the checkpoint
checkpoints = glob.glob("runs/**/*.pth.tar", recursive=True)
latest_ckpt = max(checkpoints, key=os.path.getmtime)
dst = "/kaggle/working/cifar10_resnet18_ep200_seed42.pth.tar"
shutil.copy2(latest_ckpt, dst)
print(f"Checkpoint saved: {dst}")

## CIFAR-10 Linear Evaluation

In [ ]:
result = subprocess.run([
    "python", "linear_eval.py",
    "--checkpoint", "/kaggle/working/cifar10_resnet18_ep200_seed42.pth.tar",
    "--dataset", "cifar10", "--arch", "resnet18",
    "--epochs", "100", "-b", "256", "-j", "2", "--seed", "42",
], capture_output=False)

if result.returncode != 0:
    raise RuntimeError("Linear eval failed.")

# Copy results CSV to /kaggle/working/ so it persists
if os.path.exists("linear_eval_results.csv"):
    shutil.copy2("linear_eval_results.csv", "/kaggle/working/linear_eval_results.csv")
    print("Results CSV saved.")

In [ ]:
# Print final results
import csv
with open("/kaggle/working/linear_eval_results.csv") as f:
    for row in csv.DictReader(f):
        print(f"Dataset: {row['dataset']}  |  Best Top-1: {row['best_top1']}%  |  Checkpoint: {row['checkpoint']}")